<a href="https://colab.research.google.com/github/sahithi9755/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [4]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [5]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [9]:
resume1 = """
Ravi Kumar
Email: ravi@gmail.com
Phone: 9876543210

B.Tech CSE
Aditya University
2025

Skills:
Python, SQL, Java, HTML, CSS, Git

Projects:
Library Management System

Experience:
1 year internship
"""

resume2 = """
Sneha Reddy
Email: sneha@gmail.com

B.Tech IT
JNTU
2025

Skills:
Python, SQL, Excel, Power BI, Tableau, Statistics

Projects:
Sales Dashboard

Experience:
6 months internship
"""

resume3 = """
Arun Pillai
Email: arun@gmail.com
Phone: 9999999999

B.Tech CSE
SRM University
2025

Skills:
C++, Java, Python, SQL, MongoDB, React, NodeJS, Git, Docker

Projects:
E-Commerce Platform

Experience:
1 year internship
"""

resumes = [resume1, resume2, resume3]

print("Loaded", len(resumes), "resumes")

Loaded 3 resumes


In [10]:
results = []

for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)

        results.append(parsed)

        print(
            f'\nResume {i+1}: '
            f'{parsed.name} — '
            f'{len(parsed.skills)} skills, '
            f'{parsed.experience_years} years exp'
        )

    except Exception as e:
        print(
            f'\nResume {i+1}: FAILED — '
            f'{type(e).__name__}: {str(e)[:200]}'
        )


Resume 1: Ravi Kumar — 6 skills, 1.0 years exp

Resume 2: Sneha Reddy — 6 skills, 0.5 years exp

Resume 3: Arun Pillai — 9 skills, 1.0 years exp


In [11]:
print(
    results[0].model_dump_json(indent=2)
)

{
  "name": "Ravi Kumar",
  "email": "ravi@gmail.com",
  "phone": "9876543210",
  "education": [
    {
      "degree": "B.Tech CSE",
      "institution": "Aditya University",
      "year": 2025
    }
  ],
  "skills": [
    "Python",
    "SQL",
    "Java",
    "HTML",
    "CSS",
    "Git"
  ],
  "projects": [
    "Library Management System"
  ],
  "experience_years": 1.0
}


In [12]:
try:
    bad = extract_resume('')
    print('Unexpected success')
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: ServerError
Message: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
